# Credit Default Prediction: The Cost of Explainability in Consumer Credit Risk

**Course:** COS802, MIT Big Data Science, University of Pretoria  
**Author:** <Name>, <Student number>  
**Notebook:** 02 Feature engineering

## Purpose
This notebook applies the cleaning decisions from notebook 01, builds applicant-level features from the main application table and the six supporting tables, and produces the two feature sets used for modelling: application data only, and application data combined with all supporting tables.

## Project research questions
- **RQ1** How much do bureau and repayment history tables improve default prediction over application data alone?
- **RQ2** How much predictive performance is lost when an inherently interpretable model (EBM) is used instead of LightGBM?

## Notebook outline
### 1. Setup
    1.1 Imports and data loading

### 2. Cleaning
    2.1 app_train
    2.2 prev_app
    2.3 installments

### 3. Application Features
    3.1 Domain features
    3.2 Save application-only feature set

### 4. Supporting Table Aggregates
    4.1 bureau and bureau_balance
    4.2 prev_app
    4.3 pos_cash
    4.4 credit_card
    4.5 installments

### 5. Join
    5.1 Left join onto app_train
    5.2 Validation checks

### 6. Save
    6.1 Save full feature set

### 1. Setup

1.1 Imports and loading of data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

directory = Path(r"C:\Users\mekay\Downloads\COS802 Data")
seed = 27

app_train      = pd.read_parquet(directory / "application_train.parquet")
bureau         = pd.read_parquet(directory / "bureau.parquet")
bureau_balance = pd.read_parquet(directory / "bureau_balance.parquet")
prev_app       = pd.read_parquet(directory / "previous_application.parquet")
pos_cash       = pd.read_parquet(directory / "POS_CASH_balance.parquet")
credit_card    = pd.read_parquet(directory / "credit_card_balance.parquet")
installments   = pd.read_parquet(directory / "installments_payments.parquet")

#

### 2. Setup

2.1 app_train

In [2]:
#Replace 365243 with NaN
app_train["DAYS_EMPLOYED"] = app_train["DAYS_EMPLOYED"].replace(365243, np.nan)

#Go through all columns and replace XNA and unknowns with NaN
app_train = app_train.replace(["XNA", "Unknown"], np.nan)

#Drop near-constant fields
app_train = app_train.drop(columns=["FLAG_MOBIL", "FLAG_DOCUMENT_2", "FLAG_DOCUMENT_4", "FLAG_DOCUMENT_7", "FLAG_DOCUMENT_10", 
                                    "FLAG_DOCUMENT_12", "FLAG_DOCUMENT_17", "FLAG_DOCUMENT_19", "FLAG_DOCUMENT_20", "FLAG_DOCUMENT_21"])

#

2.2 prev_app

In [3]:
#Replace 365243 with NaN for all day fields
days_cols = ["DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE", "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE", "DAYS_TERMINATION"]
prev_app[days_cols] = prev_app[days_cols].replace(365243, np.nan)

# XNA to NaN (XAP is kept as a valid category)
prev_app = prev_app.replace("XNA", np.nan)

# Drop low populated and near-constant fields
prev_app = prev_app.drop(columns=["RATE_INTEREST_PRIMARY", "RATE_INTEREST_PRIVILEGED", "FLAG_LAST_APPL_PER_CONTRACT"])

#

2.3 installments

In [4]:
# Combine partial payments into one row per instalment
installments = installments.groupby(["SK_ID_CURR", "SK_ID_PREV", "NUM_INSTALMENT_VERSION", "NUM_INSTALMENT_NUMBER"], as_index=False).agg(
    DAYS_INSTALMENT=("DAYS_INSTALMENT", "first"),
    AMT_INSTALMENT=("AMT_INSTALMENT", "first"),
    AMT_PAYMENT=("AMT_PAYMENT", "sum"),
    DAYS_ENTRY_PAYMENT=("DAYS_ENTRY_PAYMENT", "max"),
)


In [5]:
#Confirming there are now 0 duplicates
print(installments.duplicated(subset=["SK_ID_PREV", "NUM_INSTALMENT_NUMBER", "NUM_INSTALMENT_VERSION"]).sum())

0


# 

### 3. New features

3.1 Domain Features

Three ratio features are created from the application data, adapted from Koehrsen (2018): the size of the loan relative to income, the share of income needed to repay the loan, and the time spent in the current job relative to age.

In [6]:
#Credit income ratio compares the amount of debt a person has vs their income
app_train["CREDIT_INCOME_RATIO"] = app_train["AMT_CREDIT"] / app_train["AMT_INCOME_TOTAL"]


"""how long the applicant has been in their current job as a fraction of their age, for example, 2 years in a job for a 22 year old is a good record 
but is considered short"""
app_train["EMPLOYED_AGE_RATIO"] = app_train["DAYS_EMPLOYED"] / app_train["DAYS_BIRTH"]

#Percentage of loans annuity vs the persons income
app_train["ANNUITY_INCOME_RATIO"] = app_train["AMT_ANNUITY"] / app_train["AMT_INCOME_TOTAL"]

# 

3.2 Saving the new feature set

In [7]:
app_train.to_parquet(directory / "features_app.parquet")

# 

## 4. Supporting Table Aggregates

Each supporting table is aggregated to one row per applicant (`SK_ID_CURR`), so that it can be joined onto the application table. 
### 4.1 bureau and bureau_balance

`bureau_balance` holds monthly records for each bureau loan, so it is first aggregated to loan level and joined onto `bureau`. `bureau` is then aggregated to applicant level.

In [8]:
#Convert the status values to numeric, X and C values are converted to NaN
bureau_balance["STATUS_NUM"] = pd.to_numeric(bureau_balance["STATUS"], errors="coerce")

#Flag overdue months, overdue = 1 , on time = 0
bureau_balance["OVERDUE"] = np.where(bureau_balance["STATUS_NUM"] > 0, 1, 0)

#Aggregate so we have one row per loan
bb_agg = bureau_balance.groupby("SK_ID_BUREAU").agg(
    BB_MONTHS=("MONTHS_BALANCE", "count"), #Counts the number of months recorded per loan
    BB_MONTHS_OVERDUE=("OVERDUE", "sum"), #Number of months a loan was overdue
    BB_WORST_STATUS=("STATUS_NUM", "max"), #We take the worst a loan has ever defaulted as this is more relevant to banks
).reset_index()

bb_agg.head(10)

,SK_ID_BUREAU,BB_MONTHS,BB_MONTHS_OVERDUE,BB_WORST_STATUS
0,5001709,97,0,NaN
1,5001710,83,0,0.0
2,5001711,4,0,0.0
3,5001712,19,0,0.0
4,5001713,22,0,NaN
5,5001714,15,0,NaN
6,5001715,60,0,NaN
7,5001716,86,0,0.0
8,5001717,22,0,0.0
9,5001718,39,2,1.0


In [9]:
# Join this to the bureau dataset
bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")

bureau.head(10)

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY,BB_MONTHS,BB_MONTHS_OVERDUE,BB_WORST_STATUS
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.00,0.00,NaN,0.0,Consumer credit,-131,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.00,171342.00,NaN,0.0,Credit card,-20,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.50,NaN,NaN,0.0,Consumer credit,-16,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.00,NaN,NaN,0.0,Credit card,-16,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.00,NaN,NaN,0.0,Consumer credit,-21,NaN,NaN,NaN,NaN
5,215354,5714467,Active,currency 1,-273,0,27460.0,NaN,0.0,0,180000.00,71017.38,108982.62,0.0,Credit card,-31,NaN,NaN,NaN,NaN
6,215354,5714468,Active,currency 1,-43,0,79.0,NaN,0.0,0,42103.80,42103.80,0.00,0.0,Consumer credit,-22,NaN,NaN,NaN,NaN
7,162297,5714469,Closed,currency 1,-1896,0,-1684.0,-1710.0,14985.0,0,76878.45,0.00,0.00,0.0,Consumer credit,-1710,NaN,NaN,NaN,NaN
8,162297,5714470,Closed,currency 1,-1146,0,-811.0,-840.0,0.0,0,103007.70,0.00,0.00,0.0,Consumer credit,-840,NaN,NaN,NaN,NaN
9,162297,5714471,Active,currency 1,-1146,0,-484.0,NaN,0.0,0,4500.00,0.00,0.00,0.0,Credit card,-690,NaN,NaN,NaN,NaN


In [10]:
'''
Now we aggregate the data once more but on SK_ID_CURR so that each row represents one applicant with information on all their loans
Aggregates were selected for interpretability rather than generated for every column. Fields that are near-constant, mostly missing, or 
contained implausible values in notebook 01 were not aggregated.
'''

bureau_agg = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_LOAN_COUNT=("SK_ID_BUREAU", "count"), # Number of external loans
    BUREAU_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"), # Average time since loans were opened
    BUREAU_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"), # Most recent loan opened
    BUREAU_CREDIT_SUM=("AMT_CREDIT_SUM", "sum"), # Total amount borrowed
    BUREAU_DEBT_SUM=("AMT_CREDIT_SUM_DEBT", "sum"), # Total still owed
    BUREAU_OVERDUE_SUM=("AMT_CREDIT_SUM_OVERDUE", "sum"), # Total currently overdue
    BUREAU_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"), # Longest current overdue period
    BUREAU_MONTHS_OVERDUE_SUM=("BB_MONTHS_OVERDUE", "sum"), # Total months overdue across all loans
    BUREAU_WORST_STATUS=("BB_WORST_STATUS", "max"), # Worst overdue band on any loan
).reset_index()

In [11]:
'''We now create dummy variables for the CREDIT_ACTIVE categorical variable and bring in the SK_ID_CURR which we will then group and sum by to get
the total number of loan statuses per person'''

active_dummies = pd.get_dummies(bureau["CREDIT_ACTIVE"], prefix="BUREAU_ACTIVE").astype(int)
active_dummies["SK_ID_CURR"] = bureau["SK_ID_CURR"]

active_counts = active_dummies.groupby("SK_ID_CURR").sum().reset_index()

#we now merge this to the bureau_agg table
bureau_agg = bureau_agg.merge(active_counts, on="SK_ID_CURR", how="left")

In [12]:
bureau_agg.head()

,SK_ID_CURR,BUREAU_LOAN_COUNT,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_SUM,BUREAU_DEBT_SUM,BUREAU_OVERDUE_SUM,BUREAU_DAY_OVERDUE_MAX,BUREAU_MONTHS_OVERDUE_SUM,BUREAU_WORST_STATUS,BUREAU_ACTIVE_Active,BUREAU_ACTIVE_Bad debt,BUREAU_ACTIVE_Closed,BUREAU_ACTIVE_Sold
0,100001,7,-735.000000,-49,1453365.000,596686.5,0.0,0,1.0,1.0,3,0,4,0
1,100002,8,-874.000000,-103,865055.565,245781.0,0.0,0,27.0,1.0,2,0,6,0
2,100003,4,-1400.750000,-606,1017400.500,0.0,0.0,0,0.0,NaN,1,0,3,0
3,100004,2,-867.000000,-408,189037.800,0.0,0.0,0,0.0,NaN,0,0,2,0
4,100005,3,-190.666667,-62,657126.000,568408.5,0.0,0,0.0,0.0,2,0,1,0


In [13]:
#Verifying that our table has no duplicates

print(bureau_agg.shape)
print(f"Duplicate applicants: {bureau_agg['SK_ID_CURR'].duplicated().sum()}")

(305811, 14)
Duplicate applicants: 0


#

### 4.2 prev_app

Previous Home Credit applications are aggregated per applicant into counts by outcome, application and credit amounts, and timing.

In [14]:
# Aggregate previous applications to one row per applicant
prev_agg = prev_app.groupby("SK_ID_CURR").agg(
    PREV_APP_COUNT=("SK_ID_PREV", "count"), # Number of previous applications
    PREV_AMT_APPLICATION_MEAN=("AMT_APPLICATION", "mean"), # Average amount applied for
    PREV_AMT_CREDIT_MEAN=("AMT_CREDIT", "mean"), # Average amount granted
    PREV_DAYS_DECISION_MAX=("DAYS_DECISION", "max"), # Most recent application
    PREV_CNT_PAYMENT_MEAN=("CNT_PAYMENT", "mean"), # Average loan term (instalments)
).reset_index()

# Count of applications by outcome (approved, refused, canceled, unused offer)
status_dummies = pd.get_dummies(prev_app["NAME_CONTRACT_STATUS"], prefix="PREV_STATUS").astype(int)
status_dummies["SK_ID_CURR"] = prev_app["SK_ID_CURR"]
status_counts = status_dummies.groupby("SK_ID_CURR").sum().reset_index()

prev_agg = prev_agg.merge(status_counts, on="SK_ID_CURR", how="left")

# Share of previous applications that were refused
prev_agg["PREV_REFUSED_RATIO"] = prev_agg["PREV_STATUS_Refused"] / prev_agg["PREV_APP_COUNT"]

print(prev_agg.shape)
print(f"Duplicate applicants: {prev_agg['SK_ID_CURR'].duplicated().sum()}")

(338857, 11)
Duplicate applicants: 0


#

### 4.3 pos_cash

Monthly records of previous point-of-sale and cash loans are aggregated per applicant into loan counts, history length, and days-past-due behaviour.

In [15]:
# Flag months where the loan was past due
pos_cash["LATE"] = np.where(pos_cash["SK_DPD"] > 0, 1, 0)

# Aggregate to one row per applicant
pos_agg = pos_cash.groupby("SK_ID_CURR").agg(
    POS_LOAN_COUNT=("SK_ID_PREV", "nunique"), # Number of POS/cash loans
    POS_MONTHS=("MONTHS_BALANCE", "count"), # Total months of history
    POS_MONTHS_LATE=("LATE", "sum"), # Months past due
    POS_DPD_MAX=("SK_DPD", "max"), # Worst days past due
    POS_DPD_DEF_MAX=("SK_DPD_DEF", "max"), # Worst days past due, ignoring small debts
    POS_INSTALMENT_FUTURE_MEAN=("CNT_INSTALMENT_FUTURE", "mean"), # Average instalments still remaining
).reset_index()


print(pos_agg.shape)
print(f"Duplicate applicants: {pos_agg['SK_ID_CURR'].duplicated().sum()}")

(337252, 7)
Duplicate applicants: 0


# 

### 4.4 credit_card
Monthly credit card records are aggregated per applicant into balances, limits, utilisation, drawings and days-past-due behaviour.

In [16]:
# Flag months where the card was past due
credit_card["LATE"] = np.where(credit_card["SK_DPD"] > 0, 1, 0)

#Also added in the % of credit actually used
credit_card["UTILISATION"] = (credit_card["AMT_BALANCE"] / credit_card["AMT_CREDIT_LIMIT_ACTUAL"]).replace([np.inf, -np.inf], np.nan)

# Aggregate to one row per applicant
cc_agg = credit_card.groupby("SK_ID_CURR").agg(
    CC_MONTHS=("MONTHS_BALANCE", "count"), # Total months of history
    CC_BALANCE_MEAN=("AMT_BALANCE", "mean"), # Average balance
    CC_LIMIT_MEAN=("AMT_CREDIT_LIMIT_ACTUAL", "mean"), # Average credit limit
    CC_UTILISATION_MEAN=("UTILISATION", "mean"), # Average share of limit used
    CC_UTILISATION_MAX=("UTILISATION", "max"), # Highest share of limit used
    CC_DRAWINGS_MEAN=("AMT_DRAWINGS_CURRENT", "mean"), # Average monthly spending/withdrawals
    CC_MONTHS_LATE=("LATE", "sum"), # Months past due
    CC_DPD_MAX=("SK_DPD", "max"), # Worst days past due
).reset_index()

print(cc_agg.shape)
print(f"Duplicate applicants: {cc_agg['SK_ID_CURR'].duplicated().sum()}")

(103558, 9)
Duplicate applicants: 0


# 

### 4.5 installments

Using the instalment-level table from section 2.3 (partial payments already combined), each instalment is assessed for lateness and underpayment, then aggregated per applicant.

In [17]:
# Number of days late
installments["DAYS_LATE"] = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"] #If this is + then the payment was over due
installments["LATE"] = np.where(installments["DAYS_LATE"] > 0, 1, 0)

# By how much was the repayment short ?
installments["SHORTFALL"] = installments["AMT_INSTALMENT"] - installments["AMT_PAYMENT"] #If + then underpaid
installments["UNDERPAID"] = np.where(installments["SHORTFALL"] > 1, 1, 0)

# Aggregate to one row per applicant
inst_agg = installments.groupby("SK_ID_CURR").agg(
    INST_COUNT=("SK_ID_PREV", "count"), # Number of instalments
    INST_LATE_COUNT=("LATE", "sum"), # Instalments paid late
    INST_DAYS_LATE_MAX=("DAYS_LATE", "max"), # Latest any instalment was paid
    INST_DAYS_LATE_MEAN=("DAYS_LATE", "mean"), # Average days late (negative = usually early)
    INST_UNDERPAID_COUNT=("UNDERPAID", "sum"), # Instalments not paid in full
    INST_SHORTFALL_SUM=("SHORTFALL", "sum"), # Total amount underpaid
).reset_index()

print(inst_agg.shape)
print(f"Duplicate applicants: {inst_agg['SK_ID_CURR'].duplicated().sum()}")

(339587, 7)
Duplicate applicants: 0


# 

## 5. Join

### 5.1) Left join onto app_train
The aggregated supporting tables are left-joined onto the application table by `SK_ID_CURR`.  
Some applicants may have no history with the bureau, we will flag these. 

In [18]:
features_full = app_train.copy()

# Flags applicants with no bureau or credit card history
features_full["HAS_BUREAU"] = features_full["SK_ID_CURR"].isin(bureau_agg["SK_ID_CURR"]).astype(int)
features_full["HAS_CREDIT_CARD"] = features_full["SK_ID_CURR"].isin(cc_agg["SK_ID_CURR"]).astype(int)

## Left join each aggregated table
features_full = (features_full
                 .merge(bureau_agg, on="SK_ID_CURR", how="left")
                 .merge(prev_agg, on="SK_ID_CURR", how="left")
                 .merge(pos_agg, on="SK_ID_CURR", how="left")
                 .merge(cc_agg, on="SK_ID_CURR", how="left")
                 .merge(inst_agg, on="SK_ID_CURR", how="left"))

# Clean up the spaces in column names
features_full.columns = features_full.columns.str.replace(" ", "_")


# 

## 6. Save

The full feature set, combining the application features with the aggregated supporting tables, is saved for modelling in notebook 03.

In [19]:
features_full.to_parquet(directory / "features_full.parquet")